<a href="https://colab.research.google.com/github/adhishmodi/customer-churn-prediction/blob/main/notebooks/13_phase_7b_streamlit_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 7B — run the Streamlit churn prediction app in Colab

This notebook launches the Phase 7B app using the trusted Phase 7A model artifact created by Notebook 12. It does **not** retrain, tune, calibrate, or evaluate the model.

**These notebooks may use separate Colab runtimes.** Run Notebook 12, download `customer_churn_model.joblib`, then open this notebook in its own runtime and upload that file when prompted. The model artifact is intentionally not stored in Git. Only upload the artifact you generated yourself in Notebook 12: joblib uses pickle-based deserialization, so never upload an artifact from an untrusted source. Do not upload customer data that you are not authorized to process.

In [1]:
from pathlib import Path
import subprocess
import sys

from google.colab import files

REPO_URL = 'https://github.com/adhishmodi/customer-churn-prediction.git'
APP_ROOT = Path('/content/customer-churn-prediction-phase-7b')
MODEL_FILENAME = 'customer_churn_model.joblib'
ARTIFACT_PATH = APP_ROOT / 'models' / MODEL_FILENAME

if APP_ROOT.exists():
    raise FileExistsError(
        f'{APP_ROOT} already exists. Use a fresh Colab runtime or remove this '
        'temporary app directory before rerunning the setup cell.'
    )

subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', 'main', REPO_URL, str(APP_ROOT)],
    check=True,
)

print('Upload the customer_churn_model.joblib file downloaded from your own Notebook 12 run.')
uploaded = files.upload()
if set(uploaded) != {MODEL_FILENAME}:
    raise ValueError(f'Upload exactly one file named {MODEL_FILENAME}.')

artifact_bytes = uploaded[MODEL_FILENAME]
MAX_ARTIFACT_BYTES = 50 * 1024 * 1024
if not artifact_bytes or len(artifact_bytes) > MAX_ARTIFACT_BYTES:
    raise ValueError('The model artifact is empty or exceeds the 50 MB safety limit.')

ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
ARTIFACT_PATH.write_bytes(artifact_bytes)

sys.path.insert(0, str(APP_ROOT))
from src.inference import load_model_artifact
from src.preprocess import FEATURE_COLUMNS

artifact = load_model_artifact(ARTIFACT_PATH)
if artifact['calibration_method'] != 'sigmoid':
    raise ValueError('Expected the Phase 7A sigmoid-calibrated artifact.')
if list(artifact['feature_columns']) != list(FEATURE_COLUMNS):
    raise ValueError('The uploaded artifact feature schema is incompatible with this app.')

print('App source:', APP_ROOT)
print('Artifact validated: Phase 7A sigmoid calibration and 19-feature schema.')
print('Artifact bytes:', ARTIFACT_PATH.stat().st_size)


Upload the customer_churn_model.joblib file downloaded from your own Notebook 12 run.


Saving customer_churn_model.joblib to customer_churn_model.joblib
App source: /content/customer-churn-prediction-phase-7b
Artifact validated: Phase 7A sigmoid calibration and 19-feature schema.
Artifact bytes: 57872


In [2]:
import subprocess, sys

REQUIREMENTS_PATH = APP_ROOT / 'requirements.txt'
if not REQUIREMENTS_PATH.is_file():
    raise FileNotFoundError(f'Repository requirements file not found: {REQUIREMENTS_PATH}')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REQUIREMENTS_PATH), 'streamlit>=1.30'],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-r', '/content/customer-churn-prediction-phase-7b/requirements.txt', 'streamlit>=1.30'], returncode=0)

In [3]:
from google.colab import output
import time

if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()

log_path = APP_ROOT / 'streamlit.log'
log_handle = open(log_path, 'w', encoding='utf-8')
streamlit_process = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py',
     '--server.port=8501', '--server.address=0.0.0.0',
     '--server.headless=true', '--server.enableCORS=false',
     '--server.enableXsrfProtection=false'],
    cwd=APP_ROOT,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
if streamlit_process.poll() is not None:
    log_handle.flush()
    raise RuntimeError(f'Streamlit did not start. Inspect {log_path}')

print('Streamlit is running. Use the embedded app window below.')
output.serve_kernel_port_as_window(8501)


Streamlit is running. Use the embedded app window below.
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## Stop the app

Run this cell when you finish testing. Restarting the Colab runtime also stops the app.

In [4]:
if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    print('Streamlit process stopped.')
else:
    print('No running Streamlit process found.')


Streamlit process stopped.
